In [ ]:
from utils import *
import pickle
import networkx as nx

In [ ]:
with open(r"/nfs/datz/olmo_models/collapsed_all_subgraphs_postprocessed2.pkl", "rb") as input_file:
    models_data = pickle.load(input_file)

In [ ]:
models_data.keys()

In [ ]:
to_drop = {
    'step647650-tokens2715B',
    'step648650-tokens2719B',
    'step649650-tokens2723B',
    'step650650-tokens2728B',
    'step651581-tokens2731B',
}

for k in to_drop:
    models_data.pop(k, None)

In [ ]:
models_data['step10000-tokens41B']['country_capital_city'][0]["collapsed_token_subgraphs"][-1].nodes()

In [ ]:
with open(r"/nfs/datz/olmo_models/test2_collapsed_models_postprocessed.pkl", "rb") as input_file:
    test2 = pickle.load(input_file)

In [ ]:
for model_name, relations in models_data.items():
    print(f"Model: {model_name}")
    for relation_name, data_entries in relations.items():
        print(f"  Relation: {relation_name}, Number of Entries: {len(data_entries)}")

sorted_models = sorted(
        models_data.keys(), 
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )


In [ ]:
sorted_models

In [ ]:
def compute_heatmaps(models_data, sorted_models, theta=0.1):
    """
    Computes aggregated heatmaps for a set of models.
    
    - The general and entity heatmaps are aggregated over all relations.
    - The relation-answer and answer-specific heatmaps are computed separately 
      for the two relation groups (LOC and NAME).
    
    Args:
        models_data (dict): Dictionary where each key is a model name and the value is a dict
                            mapping relation names to lists of entries.
        sorted_models (list): List of model names in the desired order.
        theta (float): Threshold for binarizing the heatmaps.
        
    Returns:
        tuple: Two dictionaries:
            - heatmaps_dict_LOC: Contains the overall general and entity heatmaps (shared) plus
              the LOC-specific relation-answer heatmaps.
            - heatmaps_dict_NAME: Contains the same overall general and entity heatmaps plus
              the NAME-specific relation-answer heatmaps.
    """
    # Define the two relation groups (in lower-case)
    loc_relations = [
        "city_in_country", "company_hq", "country_capital_city",
        "food_from_country", "official_language", "plays_sport", "sights_in_city", "books_written", "company_ceo", "movie_directed"
    ]
    name_relations = [
        "books_written", "company_ceo", "movie_directed"
    ]
    
    heatmaps_dict_LOC = {}
    heatmaps_dict_NAME = {}
    
    for model_name in sorted_models:
        relations_data_model = models_data[model_name]
        print(model_name)
        
        # Initialize overall (general and entity) heatmaps and counters (for all relations)
        g_total = 0
        general_heatmap = np.zeros((32, 32), dtype=float)
        e_total = 0
        entity_heatmap = np.zeros((32, 32), dtype=float)
        
        # Initialize accumulators for relation-answer heatmaps for each group.
        relation_answer_heatmaps_LOC = {}
        relation_answer_with_specific_LOC = {}
        relation_answer_heatmaps_NAME = {}
        relation_answer_with_specific_NAME = {}
        
        # Loop through each relation in the model's data.
        for relation, entries in relations_data_model.items():
            # Process only if the relation is in one of our groups.
            if relation not in loc_relations + name_relations:
                continue
            
            # Prepare accumulators for the relation-specific heatmaps.
            relation_answer_heatmap = np.zeros((32, 32), dtype=float)
            relation_answer_total = 0
            answer_specific_heatmaps = {}
            
            # Process each entry for this relation.
            for idx, entry in enumerate(entries):
                # Ensure subj_token_span is defined.
                if entry["subj_token_span"] is None:
                    entry["subj_token_span"] = np.arange(len(entry["subject_tokens"])).tolist()
                    
                # --- Update the overall general heatmap (averaged over all entries) ---
                # Use the average over the attention head contributions.
                general_heatmap += np.mean(entry['attnheads_contribution_maps'], axis=0)
                g_total += 1
                
                # --- Update the overall entity heatmap for subject tokens ---
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in entry["subj_token_span"]:
                    one_data_map += entry['attnheads_contribution_maps'][t]
                entity_heatmap += one_data_map
                e_total += len(entry["subj_token_span"])
                
                # --- Update the overall entity and relation-answer heatmaps for answer tokens ---
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in entry["answer_token_span"]:
                    one_data_map += entry['attnheads_contribution_maps'][t - 1]
                entity_heatmap += one_data_map
                e_total += len(entry["answer_token_span"])
                relation_answer_heatmap += one_data_map
                relation_answer_total += len(entry["answer_token_span"])
                
                # --- Store answer-specific heatmap for this entry ---
                answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
                answer_specific_heatmaps[idx] = answer_specific_heatmap > theta
            
            # Normalize the relation-answer heatmap for the current relation.
            if relation_answer_total > 0:
                relation_answer_heatmap /= relation_answer_total
            else:
                relation_answer_heatmap = np.zeros((32, 32), dtype=float)
            
            # Store the thresholded relation-answer heatmap and answer-specific maps
            # in the appropriate group dictionary.
            if relation in loc_relations:
                relation_answer_heatmaps_LOC[relation] = relation_answer_heatmap > theta
                relation_answer_with_specific_LOC[relation] = answer_specific_heatmaps
            elif relation in name_relations:
                relation_answer_heatmaps_NAME[relation] = relation_answer_heatmap > theta
                relation_answer_with_specific_NAME[relation] = answer_specific_heatmaps
        
        # Normalize the overall general and entity heatmaps over all entries.
        if g_total > 0:
            general_heatmap /= g_total
        if e_total > 0:
            entity_heatmap /= e_total
        
        # Apply thresholding.
        thresholded_general = general_heatmap > theta
        thresholded_entity = entity_heatmap > theta
        
        # Store the results for the current model.
        heatmaps_dict_LOC[model_name] = {
            'general_heatmap': thresholded_general,
            'entity_heatmap': thresholded_entity,
            'relation_answer_heatmaps': relation_answer_heatmaps_LOC,
            'relation_answer_with_specific': relation_answer_with_specific_LOC
        }
        heatmaps_dict_NAME[model_name] = {
            'general_heatmap': thresholded_general,
            'entity_heatmap': thresholded_entity,
            'relation_answer_heatmaps': relation_answer_heatmaps_NAME,
            'relation_answer_with_specific': relation_answer_with_specific_NAME
        }
    
    return heatmaps_dict_LOC, heatmaps_dict_NAME


heatmaps_dict_LOC, heatmaps_dict_NAME = compute_heatmaps(models_data, sorted_models, theta=0.1)

In [ ]:
def calculate_proper_heads(heatmaps_dict):
    """
    Calculates proper entity heads, proper relation answer heads, 
    and proper answer specific heads for each model in heatmaps_dict.

    Args:
        heatmaps_dict (dict): A dictionary where each key is a model name and 
                              the value is a dictionary with heatmaps.

    Returns:
        dict: A dictionary containing proper heads for each model.
    """
    proper_heads_dict = {}

    for model_name, heatmaps in heatmaps_dict.items():
        # Extract heatmaps
        general_heads = heatmaps['general_heatmap']
        entity_heads = heatmaps['entity_heatmap']
        relation_answer_heads = heatmaps['relation_answer_heatmaps']
        answer_specific_heads = heatmaps['relation_answer_with_specific']

        # Calculate proper heads
        proper_entity_heads = np.logical_and(entity_heads, np.logical_not(general_heads))
        
        # Initialize dictionaries to store results for relations and specific answers
        proper_relation_answer_heads = {}
        proper_answer_specific_heads = {}

        # Calculate proper relation answer heads per relation
        for relation, relation_heads in relation_answer_heads.items():
            proper_relation_answer_heads[relation] = np.logical_and(
                relation_heads,
                np.logical_not(np.logical_or(proper_entity_heads, general_heads))
            )

        # Calculate proper answer specific heads per relation and specific answer
        for relation, specific_answers in answer_specific_heads.items():
            proper_answer_specific_heads[relation] = {}
            for answer_id, specific_heads in specific_answers.items():
                proper_answer_specific_heads[relation][answer_id] = np.logical_and(
                    specific_heads,
                    np.logical_not(
                        np.logical_or(
                            proper_relation_answer_heads[relation],
                            np.logical_or(proper_entity_heads, general_heads)
                        )
                    )
                )

        # Store results in the output dictionary
        proper_heads_dict[model_name] = {
            'proper_general_heads': general_heads,
            'proper_entity_heads': proper_entity_heads,
            'proper_relation_answer_heads': proper_relation_answer_heads,
            'proper_answer_specific_heads': proper_answer_specific_heads
        }

    return proper_heads_dict

# Now, assuming you have built two separate dictionaries:
#   heatmaps_dict_LOC and heatmaps_dict_NAME
# you can calculate the proper heads for each group as follows:

proper_heads_LOC = calculate_proper_heads(heatmaps_dict_LOC)
proper_heads_NAME = calculate_proper_heads(heatmaps_dict_NAME)

In [ ]:
def calculate_consistency_proper_heads(heatmaps_dict, relation_name, fact_idx):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        #if model_name == 'main':
        #    continue

        # Store IoU results for this model
        model_ious = {}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]
            elif heatmap_type == "proper_answer_specific_heads":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
            elif heatmap_type == "proper_entity_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            elif heatmap_type == "proper_general_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            else:
                continue

            # Calculate IoU
            intersection = np.logical_and(main_heatmap, model_heatmap)
            union = np.logical_or(main_heatmap, model_heatmap)

            area_of_intersection = np.sum(intersection)
            area_of_union = np.sum(union)

            # Avoid division by zero
            iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

            # Store IoU result and statistics
            model_ious[heatmap_type] = {
                'iou': iou,
                'head_count': model_heatmap.sum(),
                'intersection': area_of_intersection,
                'union': area_of_union
            }

        # Add this model's IoU results to the main dictionary
        iou_results[model_name] = model_ious

    return iou_results

In [ ]:
def plot_proper_heads_iou_multiple(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []
    
    general_head_count = []
    entity_head_count = []
    answer_all_head_count = []
    answer_head_counts = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])
        
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])

    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_ious, label="Proper General Heads IoU", marker='o', linestyle='-')
    plt.plot(model_names, entity_ious, label="Proper Entity Heads IoU", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_ious, label="Proper Relation Answer Heads IoU", marker='x', linestyle='-.')
    plt.plot(model_names, answer_ious, label="Proper Answer Specific Heads IoU", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("IoU")
    plt.title(f"Proper Heads IoU Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_head_count, label="Proper General Heads Count", marker='o', linestyle='-')
    plt.plot(model_names, entity_head_count, label="Proper Entity Heads Count", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_head_count, label="Proper Relation Answer Heads Count", marker='x', linestyle='-.')
    plt.plot(model_names, answer_head_counts, label="Proper Answer Specific Heads Count", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("IoU")
    plt.title(f"Proper Heads IoU Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
def plot_proper_heads_counts_intersection(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )

    general_intersections = []
    entity_intersections = []
    answer_all_intersections = []
    answer_intersections = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_intersections.append(iou_results[model_name]['proper_general_heads']['intersection'])
        entity_intersections.append(iou_results[model_name]['proper_entity_heads']['intersection'])
        answer_all_intersections.append(iou_results[model_name]['proper_relation_answer_heads']['intersection'])
        answer_intersections.append(iou_results[model_name]['proper_answer_specific_heads']['intersection'])

    # Plotting Intersections
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_intersections, label="Proper General Heads Intersections", marker='o', linestyle='-')
    plt.plot(model_names, entity_intersections, label="Proper Entity Heads Intersections", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_intersections, label="Proper Relation Answer Heads Intersections", marker='x', linestyle='-.')
    plt.plot(model_names, answer_intersections, label="Proper Answer Specific Heads Intersections", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Intersection Counts")
    plt.title(f"Proper Heads Intersections Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_proper_heads_counts_union(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )

    general_unions = []
    entity_unions = []
    answer_all_unions = []
    answer_unions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_unions.append(iou_results[model_name]['proper_general_heads']['union'])
        entity_unions.append(iou_results[model_name]['proper_entity_heads']['union'])
        answer_all_unions.append(iou_results[model_name]['proper_relation_answer_heads']['union'])
        answer_unions.append(iou_results[model_name]['proper_answer_specific_heads']['union'])

    # Plotting Intersections
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_unions, label="Proper General Heads Unions", marker='o', linestyle='-')
    plt.plot(model_names, entity_unions, label="Proper Entity Heads Unions", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_unions, label="Proper Relation Answer Heads Unions", marker='x', linestyle='-.')
    plt.plot(model_names, answer_unions, label="Proper Answer Specific Heads Unions", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Union Counts")
    plt.title(f"Proper Heads Unions Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
def calculate_consistency_proper_heads_all_facts(heatmaps_dict, relation_name):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    for model_name, model_data in heatmaps_dict.items():
        #if model_name == 'main':
        #    continue

        # Store IoU, intersection, and union results for this model
        model_metrics = {heatmap_type: {'iou': [], 'head_count': [], 'intersection': [], 'union': []}
                         for heatmap_type in main_heatmaps.keys()}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Iterate over all facts (assume array structure)
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            elif heatmap_type == "proper_answer_specific_heads":
                # Iterate over all facts and answers
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                    #for answer_id, main_specific_heatmap in enumerate(
                    #    main_heatmaps[heatmap_type][relation_name][fact_idx]
                    #~):
                main_specific_heatmap = np.logical_or.reduce(list(main_heatmaps[heatmap_type][relation_name].values())) #main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_specific_heatmap = np.logical_or.reduce(list(model_data[heatmap_type][relation_name].values())) #model_data[heatmap_type][relation_name][fact_idx]#[answer_id]

                # Calculate metrics
                intersection = np.logical_and(main_specific_heatmap, model_specific_heatmap)
                union = np.logical_or(main_specific_heatmap, model_specific_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_specific_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            else:
                # Handle proper_entity_heads and proper_general_heads
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

        # Aggregate results for each heatmap type
        aggregated_metrics = {ht: {
            'iou': np.mean(metrics['iou']),
            'head_count': np.sum(metrics['head_count']),
            'intersection': np.sum(metrics['intersection']),
            'union': np.sum(metrics['union'])
        } for ht, metrics in model_metrics.items()}

        iou_results[model_name] = aggregated_metrics

    return iou_results


def plot_aggregated_proper_heads_iou(iou_results, relation_name, output_dir):
    # Prepare data
    sorted_model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    
    # Create abbreviated model names using the helper function.
    abbreviated_model_names = [
        abbreviate_model_name(model, i) for i, model in enumerate(sorted_model_names, start=1)
    ]
    
    # For the IoU plot, exclude the 'main' snapshot if it is the last element.
    if sorted_model_names[-1] == 'main':
        sorted_model_names_iou = sorted_model_names[:-1]
        abbreviated_model_names_iou = abbreviated_model_names[:-1]
    else:
        sorted_model_names_iou = sorted_model_names
        abbreviated_model_names_iou = abbreviated_model_names
        
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []

    general_head_count = []
    entity_head_count = []
    answer_all_head_count = []
    answer_head_counts = []

    # Populate lists for each heatmap type
    for model_name in sorted_model_names_iou:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])

    for model_name in sorted_model_names:
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])

    # Create directories if not exist
    iou_dir = os.path.join(output_dir, "iou")
    count_dir = os.path.join(output_dir, "count")
    os.makedirs(iou_dir, exist_ok=True)
    os.makedirs(count_dir, exist_ok=True)

    # Plot and save IoU
    plt.figure(figsize=(20, 6))
    plt.plot(abbreviated_model_names_iou, general_ious, label="Proper General Heads IoU", marker='o', linestyle='-')
    plt.plot(abbreviated_model_names_iou, entity_ious, label="Proper Entity Heads IoU", marker='s', linestyle='--')
    plt.plot(abbreviated_model_names_iou, answer_all_ious, label="Proper Relation Answer Heads IoU", marker='x', linestyle='-.')
    plt.plot(abbreviated_model_names_iou, answer_ious, label="Proper Answer Specific Heads IoU", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(abbreviated_model_names_iou)), labels=abbreviated_model_names_iou, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated IoU")
    plt.title(f"Aggregated Proper Heads IoU Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    plt.savefig(os.path.join(iou_dir, f"{relation_name}_heads_iou.pdf"), dpi=500, bbox_inches='tight', format="pdf")
    plt.close()

    # Plot and save Count
    plt.figure(figsize=(20, 6))
    plt.plot(abbreviated_model_names, general_head_count, label="Proper General Heads Count", marker='o', linestyle='-')
    plt.plot(abbreviated_model_names, entity_head_count, label="Proper Entity Heads Count", marker='s', linestyle='--')
    plt.plot(abbreviated_model_names, answer_all_head_count, label="Proper Relation Answer Heads Count", marker='x', linestyle='-.')
    plt.plot(abbreviated_model_names, answer_head_counts, label="Proper Answer Specific Heads Count", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(abbreviated_model_names)), labels=abbreviated_model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated Count")
    plt.title(f"Aggregated Proper Heads Count Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(count_dir, f"{relation_name}_heads_count.pdf"), dpi=500, bbox_inches='tight', format="pdf")
    plt.show()
    plt.close()

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt

def abbreviate_model_name(model_name, index):
    """
    Example helper function to abbreviate model names, e.g. "step10000-tokens41B" -> "S1-41B".
    Adjust to match your existing abbreviation logic.
    """
    match = re.search(r'tokens(\d+B)', model_name)
    token_part = match.group(1) if match else ""
    if "main" in model_name:
        return "Main"
    return f"S{index}-{token_part}"

def plot_aggregated_proper_heads_iou_and_count(iou_results, relation_name, output_dir):
    # -----------------------------
    # Prepare data & model ordering
    # -----------------------------
    # Sort models by numeric step, keeping 'main' last if present
    sorted_model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )

    # Create abbreviated model names
    abbreviated_model_names = [
        abbreviate_model_name(model, i) for i, model in enumerate(sorted_model_names, start=1)
    ]

    # If 'main' is last, for IoU we might exclude it (as in your original code).
    if sorted_model_names[-1] == 'main':
        sorted_model_names_iou = sorted_model_names[:-1]
        abbreviated_model_names_iou = abbreviated_model_names[:-1]
    else:
        sorted_model_names_iou = sorted_model_names
        abbreviated_model_names_iou = abbreviated_model_names

    # -----------------------------
    # Extract IoU data
    # -----------------------------
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []

    for model_name in sorted_model_names_iou:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])

    # -----------------------------
    # Extract Count data
    # -----------------------------
    general_head_count = []
    entity_head_count = []
    answer_all_head_count = []
    answer_head_counts = []

    for model_name in sorted_model_names:
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])

    # -----------------------------
    # Create output directories (optional)
    # -----------------------------
    iou_dir = os.path.join(output_dir, "iou")
    count_dir = os.path.join(output_dir, "count")
    os.makedirs(iou_dir, exist_ok=True)
    os.makedirs(count_dir, exist_ok=True)

    # -----------------------------
    # Combine plots into one figure
    # -----------------------------
    fig, (ax_iou, ax_count) = plt.subplots(nrows=1, ncols=2, figsize=(24, 6))

    # === Left subplot: IoU ===
    ax_iou.plot(abbreviated_model_names_iou, general_ious,
                label="Proper General Heads IoU", marker='o', linestyle='-')
    ax_iou.plot(abbreviated_model_names_iou, entity_ious,
                label="Proper Entity Heads IoU", marker='s', linestyle='--')
    ax_iou.plot(abbreviated_model_names_iou, answer_all_ious,
                label="Proper Relation Answer Heads IoU", marker='x', linestyle='-.')
    ax_iou.plot(abbreviated_model_names_iou, answer_ious,
                label="Proper Answer Specific Heads IoU", marker='^', linestyle='-.')
    ticks = range(0, len(abbreviated_model_names_iou), 5)
    labels = [abbreviated_model_names_iou[i] for i in ticks]

    ax_iou.set_xticks(ticks)
    ax_iou.set_xticklabels(labels, rotation=45, ha='right')
    ax_iou.set_xlabel("Model Steps", fontsize=14)
    ax_iou.set_ylabel("Aggregated IoU", fontsize=14)
    ax_iou.set_title(f"Aggregated Proper Heads IoU\nRelation: {relation_name}", fontsize=16)
    ax_iou.tick_params(axis='both', labelsize=12)  # Change tick label size
    ax_iou.legend(fontsize=12)
    ax_iou.grid(True)

    # === Right subplot: Count ===
    ax_count.plot(abbreviated_model_names, general_head_count,
                  label="Proper General Heads Count", marker='o', linestyle='-')
    ax_count.plot(abbreviated_model_names, entity_head_count,
                  label="Proper Entity Heads Count", marker='s', linestyle='--')
    ax_count.plot(abbreviated_model_names, answer_all_head_count,
                  label="Proper Relation Answer Heads Count", marker='x', linestyle='-.')
    ax_count.plot(abbreviated_model_names, answer_head_counts,
                  label="Proper Answer Specific Heads Count", marker='^', linestyle='-.')

    plt.xticks(ticks=range(0, len(abbreviated_model_names), 5), labels=abbreviated_model_names[::5], rotation=45, ha='right')
    ax_count.set_xticklabels(abbreviated_model_names[::5], rotation=45, ha='right')
    ax_count.set_xlabel("Model Steps", fontsize=14)
    ax_count.set_ylabel("Aggregated Count", fontsize=14)
    ax_count.set_title(f"Aggregated Proper Heads Count\nRelation: {relation_name}", fontsize=16)
    ax_count.tick_params(axis='both', labelsize=12)
    ax_count.legend(fontsize=12)

    ax_count.grid(True)

    # Adjust spacing & save
    plt.tight_layout()
    # Save a single PDF (both subplots)
    combined_path = os.path.join(output_dir, f"{relation_name}_combined_iou_count.pdf")
    #plt.savefig(combined_path, dpi=500, bbox_inches='tight', format="pdf")

    # Show the combined figure
    plt.show()
    plt.close(fig)


In [ ]:
loc_relations = [
    "city_in_country", "company_hq", "country_capital_city",
    "food_from_country", "official_language", "plays_sport", "sights_in_city", "books_written", "company_ceo", "movie_directed"
]
name_relations = ["books_written", "company_ceo", "movie_directed"]

In [ ]:
relation_names

In [ ]:
relation_names = models_data['main'].keys()
output_dir = "final_figures2"
for relation_name in loc_relations:
    # Extract the sentences for all facts (optional, for reference)
    sentences = [models_data['main'][relation_name][fact_idx]['sentence'] for fact_idx in range(len(models_data['main'][relation_name]))]

    # Step 1: Calculate proper heads
    proper_heads = calculate_proper_heads(heatmaps_dict_LOC)

    # Step 2: Calculate overlap (IoU) across all facts for the relation
    overlap = calculate_consistency_proper_heads_all_facts(proper_heads, relation_name)

    # Step 3: Plot aggregated IoU across all facts for the relation
    plot_aggregated_proper_heads_iou_and_count(overlap, relation_name, output_dir)

In [ ]:
heatmaps_dict_NAME

In [ ]:
def compute_heatmaps_agg(models_data, sorted_models, theta=0.1):
    """
    Computes aggregated heatmaps for a set of models.
    
    - The overall general and entity heatmaps are aggregated over all relations.
    - The relation-answer and answer-specific heatmaps are computed separately 
      for the two relation groups (LOC and NAME) and then aggregated over all relations
      within each group.
    
    Args:
        models_data (dict): Dictionary where each key is a model name and the value is a dict
                            mapping relation names to lists of entries.
        sorted_models (list): List of model names in the desired order.
        theta (float): Threshold for binarizing the heatmaps.
        
    Returns:
        tuple: Two dictionaries:
            - heatmaps_dict_LOC: For each model, contains the overall general and entity heatmaps (shared)
              plus an aggregated relation-answer heatmap and aggregated answer-specific heatmap for LOC relations.
            - heatmaps_dict_NAME: Similarly, for NAME relations.
    """
    # Define the two relation groups (in lower-case)
    loc_relations = [
        "city_in_country", "company_hq", "country_capital_city",
        "food_from_country", "official_language", "plays_sport", "sights_in_city","books_written", "company_ceo", "movie_directed"
    ]
    name_relations = [
        "books_written", "company_ceo", "movie_directed"
    ]
    
    heatmaps_dict_LOC = {}
    heatmaps_dict_NAME = {}
    
    for model_name in sorted_models:
        relations_data_model = models_data[model_name]
        print(model_name)
        
        # --- Overall (general and entity) heatmaps: computed over all relations ---
        g_total = 0
        general_heatmap = np.zeros((32, 32), dtype=float)
        e_total = 0
        entity_heatmap = np.zeros((32, 32), dtype=float)
        
        # --- Aggregators for relation-answer and answer-specific heatmaps per group ---
        # For LOC
        aggregated_relation_answer_heatmap_LOC = np.zeros((32, 32), dtype=bool)
        aggregated_answer_specific_heatmap_LOC = np.zeros((32, 32), dtype=bool)
        # For NAME
        aggregated_relation_answer_heatmap_NAME = np.zeros((32, 32), dtype=bool)
        aggregated_answer_specific_heatmap_NAME = np.zeros((32, 32), dtype=bool)
        
        # Process each relation in the model's data.
        for relation, entries in relations_data_model.items():
            # Process only if the relation is in one of our groups.
            if relation not in loc_relations + name_relations:
                continue
            
            # Prepare accumulators for the current relation.
            relation_answer_heatmap = np.zeros((32, 32), dtype=float)
            relation_answer_total = 0
            # We'll also store answer-specific heatmaps in a list so that we can aggregate them.
            answer_specific_list = []
            
            for idx, entry in enumerate(entries):
                # Ensure subject token span is defined.
                if entry["subj_token_span"] is None:
                    entry["subj_token_span"] = np.arange(len(entry["subject_tokens"])).tolist()
                    
                # --- Update overall general heatmap (averaged over all entries) ---
                general_heatmap += np.mean(entry['attnheads_contribution_maps'], axis=0)
                g_total += 1
                
                # --- Update overall entity heatmap for subject tokens ---
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in entry["subj_token_span"]:
                    one_data_map += entry['attnheads_contribution_maps'][t]
                entity_heatmap += one_data_map
                e_total += len(entry["subj_token_span"])
                
                # --- Update overall entity and relation-answer heatmaps for answer tokens ---
                one_data_map = np.zeros((32, 32), dtype=float)
                for t in entry["answer_token_span"]:
                    one_data_map += entry['attnheads_contribution_maps'][t - 1]
                entity_heatmap += one_data_map
                e_total += len(entry["answer_token_span"])
                relation_answer_heatmap += one_data_map
                relation_answer_total += len(entry["answer_token_span"])
                
                # --- Compute answer-specific heatmap for this entry ---
                answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
                answer_specific_list.append(answer_specific_heatmap > theta)
            
            # Normalize the relation-answer heatmap for the current relation.
            if relation_answer_total > 0:
                relation_answer_heatmap /= relation_answer_total
            else:
                relation_answer_heatmap = np.zeros((32, 32), dtype=float)
            
            # Threshold the relation-answer heatmap.
            relation_answer_thresh = relation_answer_heatmap > theta
            
            # Aggregate this relation's heatmaps into the corresponding group accumulator.
            if relation in loc_relations:
                aggregated_relation_answer_heatmap_LOC = np.logical_or(
                    aggregated_relation_answer_heatmap_LOC, relation_answer_thresh)
                # Aggregate each answer-specific heatmap.
                for heat in answer_specific_list:
                    aggregated_answer_specific_heatmap_LOC = np.logical_or(
                        aggregated_answer_specific_heatmap_LOC, heat)
            elif relation in name_relations:
                aggregated_relation_answer_heatmap_NAME = np.logical_or(
                    aggregated_relation_answer_heatmap_NAME, relation_answer_thresh)
                for heat in answer_specific_list:
                    aggregated_answer_specific_heatmap_NAME = np.logical_or(
                        aggregated_answer_specific_heatmap_NAME, heat)
        
        # Normalize overall general and entity heatmaps.
        if g_total > 0:
            general_heatmap /= g_total
        if e_total > 0:
            entity_heatmap /= e_total
        
        # Apply thresholding to overall heatmaps.
        thresholded_general = general_heatmap > theta
        thresholded_entity = entity_heatmap > theta
        
        # Store the results for the current model.
        heatmaps_dict_LOC[model_name] = {
            'general_heatmap': thresholded_general,
            'entity_heatmap': thresholded_entity,
            'relation_answer_heatmaps': aggregated_relation_answer_heatmap_LOC,
            'relation_answer_with_specific': aggregated_answer_specific_heatmap_LOC
        }
        heatmaps_dict_NAME[model_name] = {
            'general_heatmap': thresholded_general,
            'entity_heatmap': thresholded_entity,
            'relation_answer_heatmaps': aggregated_relation_answer_heatmap_NAME,
            'answer_specific_heatmap': aggregated_answer_specific_heatmap_NAME
        }
    
    return heatmaps_dict_LOC, heatmaps_dict_NAME

heatmaps_dict_LOC_agg, _ = compute_heatmaps_agg(models_data, sorted_models, theta=0.1)

In [ ]:
def calculate_proper_heads_agg(heatmaps_dict):
    """
    Calculates proper entity heads, proper relation answer heads, 
    and proper answer specific heads for each model in heatmaps_dict.
    
    This version is designed for the new heatmap structure, where for each model
    the dictionary contains:
        - 'general_heatmap': overall general heatmap (Boolean array)
        - 'entity_heatmap': overall entity heatmap (Boolean array)
        - 'relation_answer_heatmap': aggregated relation-answer heatmap (Boolean array)
        - 'answer_specific_heatmap': aggregated answer-specific heatmap (Boolean array)
    
    The proper heads are defined as:
      - proper_entity_heads: heads active in the entity heatmap but not in the general heatmap.
      - proper_relation_answer_heads: heads in the relation-answer heatmap but not in the union 
        of the general and proper_entity_heads.
      - proper_answer_specific_heads: heads in the answer-specific heatmap but not in the union 
        of proper_relation_answer_heads, proper_entity_heads, and general heatmap.
    
    Args:
        heatmaps_dict (dict): A dictionary where each key is a model name and the value is a dictionary
                              containing the four heatmaps.
    
    Returns:
        dict: A dictionary containing proper heads for each model.
    """
    proper_heads_dict = {}

    for model_name, heatmaps in heatmaps_dict.items():
        # Extract aggregated heatmaps
        general_heads = heatmaps['general_heatmap']
        entity_heads = heatmaps['entity_heatmap']
        relation_answer_heatmap = heatmaps['relation_answer_heatmaps']
        answer_specific_heatmap = heatmaps['relation_answer_with_specific']

        # Compute proper entity heads: those active in the entity heatmap but not in the general heatmap.
        proper_entity_heads = np.logical_and(entity_heads, np.logical_not(general_heads))
        
        # Compute proper relation answer heads: those active in the relation-answer heatmap,
        # but not present in the union of general and proper entity heads.
        proper_relation_answer_heads = np.logical_and(
            relation_answer_heatmap,
            np.logical_not(np.logical_or(proper_entity_heads, general_heads))
        )
        
        # Compute proper answer specific heads: those active in the answer-specific heatmap,
        # but not present in the union of proper relation-answer, proper entity, and general heads.
        proper_answer_specific_heads = np.logical_and(
            answer_specific_heatmap,
            np.logical_not(
                np.logical_or(
                    proper_relation_answer_heads,
                    np.logical_or(proper_entity_heads, general_heads)
                )
            )
        )

        proper_heads_dict[model_name] = {
            'proper_general_heads': general_heads,
            'proper_entity_heads': proper_entity_heads,
            'proper_relation_answer_heads': proper_relation_answer_heads,
            'proper_answer_specific_heads': proper_answer_specific_heads
        }

    return proper_heads_dict

proper_heads_LOC_agg = calculate_proper_heads_agg(heatmaps_dict_LOC_agg)

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt


def calculate_consistency_proper_heads_all_facts_agg(group_proper_heads_dict):
    """
    Calculates IoU (Intersection over Union) between the proper heads of each model 
    (compared to the 'main' model) for a given relation group (e.g. LOC or NAME),
    aggregated over all relations.

    Args:
        group_proper_heads_dict (dict): A dictionary where each key is a model name and 
                                        the value is a dictionary with the following keys:
                                        'general_heatmap',
                                        'entity_heatmap',
                                        'relation_answer_heatmap',
                                        'answer_specific_heatmap'.

    Returns:
        dict: A dictionary containing IoU and head statistics for each model,
              aggregated over all facts.
    """
    iou_results = {}
    
    # Retrieve the 'main' model's aggregated heatmaps.
    main_heads = group_proper_heads_dict['main']
    
    for model_name, model_data in group_proper_heads_dict.items():
        # Initialize a dictionary to accumulate metrics per heatmap type.
        model_metrics = {
            ht: {'iou': [], 'head_count': [], 'intersection': [], 'union': []}
            for ht in main_heads.keys()
        }
        
        for heatmap_type in main_heads.keys():
            main_heatmap = main_heads[heatmap_type]
            model_heatmap = model_data[heatmap_type]
            
            # Calculate intersection and union.
            intersection = np.logical_and(main_heatmap, model_heatmap)
            union = np.logical_or(main_heatmap, model_heatmap)
            area_of_intersection = np.sum(intersection)
            area_of_union = np.sum(union)
            iou = area_of_intersection / area_of_union if area_of_union > 0 else 0
            
            # Accumulate metrics.
            model_metrics[heatmap_type]['iou'].append(iou)
            model_metrics[heatmap_type]['head_count'].append(np.sum(model_heatmap))
            model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
            model_metrics[heatmap_type]['union'].append(area_of_union)
        
        # Aggregate results for each heatmap type.
        aggregated_metrics = {
            ht: {
                'iou': np.mean(metrics['iou']),
                'head_count': np.sum(metrics['head_count']),
                'intersection': np.sum(metrics['intersection']),
                'union': np.sum(metrics['union'])
            }
            for ht, metrics in model_metrics.items()
        }
        iou_results[model_name] = aggregated_metrics

    return iou_results


def plot_aggregated_proper_heads_iou_agg(iou_results, output_dir):
    """
    Plots aggregated IoU and head count metrics for proper heads across models 
    for a given relation group (e.g. LOC or NAME).

    Args:
        iou_results (dict): The output from calculate_consistency_proper_heads_all_facts,
                            keyed by model name.
        output_dir (str): Directory in which to save the plots.
    """
    # Sort model names by training step (with 'main' last)
    sorted_model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    
    # Create abbreviated model names using the helper function.
    abbreviated_model_names = [
        abbreviated_model_name(model, i) for i, model in enumerate(sorted_model_names, start=1)
    ]
    
    # For the IoU plot, exclude the 'main' snapshot if it is the last element.
    if sorted_model_names[-1] == 'main':
        sorted_model_names_iou = sorted_model_names[:-1]
        abbreviated_model_names_iou = abbreviated_model_names[:-1]
    else:
        sorted_model_names_iou = sorted_model_names
        abbreviated_model_names_iou = abbreviated_model_names
    
    general_ious = []
    entity_ious = []
    relation_answer_ious = []
    answer_specific_ious = []
    
    general_head_count = []
    entity_head_count = []
    relation_answer_head_count = []
    answer_specific_head_count = []
    
    # Populate lists (for IoU, we ignore the main model when plotting if desired)
    for model_name in sorted_model_names_iou:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        relation_answer_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_specific_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])
    
    for model_name in sorted_model_names:
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        relation_answer_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_specific_head_count.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])
    
    # Create directories if they do not exist.
    iou_dir = os.path.join(output_dir, "iou")
    count_dir = os.path.join(output_dir, "count")
    os.makedirs(iou_dir, exist_ok=True)
    os.makedirs(count_dir, exist_ok=True)
    
    # Plot and show aggregated IoU.
    plt.figure(figsize=(20, 6))
    plt.plot(abbreviated_model_names_iou, general_ious, label="Proper General Heads IoU (Aggregated)", marker='o', linestyle='-')
    plt.plot(abbreviated_model_names_iou, entity_ious, label="Proper Entity Heads IoU (Aggregated)", marker='s', linestyle='--')
    plt.plot(abbreviated_model_names_iou, relation_answer_ious, label="Proper Relation Answer Heads IoU (Aggregated)", marker='x', linestyle='-.')
    plt.plot(abbreviated_model_names_iou, answer_specific_ious, label="Proper Answer Specific Heads IoU (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(abbreviated_model_names_iou)), labels=abbreviated_model_names_iou, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated IoU")
    plt.title("Aggregated Proper Heads IoU Across Models")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(iou_dir, "aggregated_heads_iou.pdf"), dpi=500, bbox_inches='tight', format="pdf")
    plt.show()
    # Optionally, save the figure:
    plt.close()
    
    # Plot and show aggregated head counts.
    plt.figure(figsize=(20, 6))
    plt.plot(abbreviated_model_names, general_head_count, label="Proper General Heads Count (Aggregated)", marker='o', linestyle='-')
    plt.plot(abbreviated_model_names, entity_head_count, label="Proper Entity Heads Count (Aggregated)", marker='s', linestyle='--')
    plt.plot(abbreviated_model_names, relation_answer_head_count, label="Proper Relation Answer Heads Count (Aggregated)", marker='x', linestyle='-.')
    plt.plot(abbreviated_model_names, answer_specific_head_count, label="Proper Answer Specific Heads Count (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(abbreviated_model_names)), labels=abbreviated_model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated Head Count")
    plt.title("Aggregated Proper Heads Count Across Models")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(count_dir, "aggregated_heads_count.pdf", dpi=500, bbox_inches='tight', format="pdf"))
    plt.show()
    # Optionally, save the figure:
    plt.close()

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt

# Make sure these helper functions are defined somewhere in your code.
def extract_step_number(model_name):
    """
    Extracts the step number from the model name using regex.
    'main' is assigned an infinite step so that it comes last.
    """
    if model_name == 'main':
        return float('inf')
    match = re.search(r'\d+', model_name)
    return int(match.group()) if match else float('inf')

def abbreviate_model_name(model_name, index):
    """
    Converts a model name like 'step10000-tokens41B' into a compact label 'S{index}-41B'.
    For example, if index is 1 and model_name is 'step10000-tokens41B', then the label
    will be 'S1-41B'. For 'main', we simply return 'Main'.
    """
    if model_name == 'main':
        return "Main"
    token_match = re.search(r'tokens(\d+B)', model_name)
    token_part = token_match.group(1) if token_match else ""
    return f"S{index}-{token_part}"


def plot_aggregated_proper_heads_iou_agg(iou_results, output_dir):
    """
    Plots aggregated IoU and head count metrics for proper heads across models 
    for a given relation group (e.g. LOC or NAME), using abbreviated snapshot names
    for the x-axis.

    Args:
        iou_results (dict): The output from calculate_consistency_proper_heads_all_facts_agg,
                            keyed by model name.
        output_dir (str): Directory in which to save the plots.
    """
    # Sort model names by training step (with 'main' last)
    model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    
    # Create abbreviated model names for x-axis labels.
    abbreviated_model_names = [abbreviate_model_name(name, idx) 
                               for idx, name in enumerate(model_names, start=1)]
    
    # For the IoU plot, exclude the 'main' model (assumed to be the last element)
    if model_names[-1] == 'main':
        model_names_iou = model_names[:-1]
        abbreviated_model_names_iou = abbreviated_model_names[:-1]
    else:
        model_names_iou = model_names
        abbreviated_model_names_iou = abbreviated_model_names

    # Initialize lists for IoU metrics.
    general_ious = []
    entity_ious = []
    relation_answer_ious = []
    answer_specific_ious = []
    
    # Initialize lists for head counts.
    general_head_count = []
    entity_head_count = []
    relation_answer_head_count = []
    answer_specific_head_count = []
    
    # Populate lists for IoU (skip 'main')
    for model_name in model_names_iou:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        relation_answer_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_specific_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])
    
    # Populate lists for head counts (include all models)
    for model_name in model_names:
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        relation_answer_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_specific_head_count.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])
    
    # Create directories if they do not exist.
    iou_dir = os.path.join(output_dir, "iou")
    count_dir = os.path.join(output_dir, "count")
    os.makedirs(iou_dir, exist_ok=True)
    os.makedirs(count_dir, exist_ok=True)
    
    # ------------------------------
    # Plot aggregated IoU.
    # ------------------------------
    plt.figure(figsize=(20, 6))
    plt.plot(abbreviated_model_names_iou, general_ious, label="Proper General Heads IoU", marker='o', linestyle='-')
    plt.plot(abbreviated_model_names_iou, entity_ious, label="Proper Entity Heads IoU", marker='s', linestyle='--')
    plt.plot(abbreviated_model_names_iou, relation_answer_ious, label="Proper Relation Answer Heads IoU", marker='x', linestyle='-.')
    plt.plot(abbreviated_model_names_iou, answer_specific_ious, label="Proper Answer Specific Heads IoU", marker='^', linestyle='-.')

    plt.xticks(rotation=45, ha='right')
    plt.xlabel("Model Snapshots")
    plt.ylabel("Aggregated IoU")
    plt.title("Aggregated Proper Heads IoU Across Models")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    # Optionally, save the figure:
    plt.savefig(os.path.join(iou_dir, "aggregated_heads_iou.pdf"), dpi=500, bbox_inches='tight', format="pdf")
    plt.show()
    plt.close()
    
    # ------------------------------
    # Plot aggregated head counts.
    # ------------------------------
    plt.figure(figsize=(20, 6))
    plt.plot(abbreviated_model_names, general_head_count, label="Proper General Heads Count", marker='o', linestyle='-')
    plt.plot(abbreviated_model_names, entity_head_count, label="Proper Entity Heads Count", marker='s', linestyle='--')
    plt.plot(abbreviated_model_names, relation_answer_head_count, label="Proper Relation Answer Heads Count", marker='x', linestyle='-.')
    plt.plot(abbreviated_model_names, answer_specific_head_count, label="Proper Answer Specific Heads Count", marker='^', linestyle='-.')

    plt.xticks(rotation=45, ha='right')
    plt.xlabel("Model Snapshots")
    plt.ylabel("Aggregated Head Count")
    plt.title("Aggregated Proper Heads Count Across Models")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    # Optionally, save the figure:
    plt.savefig(os.path.join(count_dir, "aggregated_heads_count.pdf"), dpi=500, bbox_inches='tight', format="pdf")
    plt.show()
    plt.close()

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt

def extract_step_number(model_name):
    """
    Extracts the step number from the model name using regex.
    'main' is assigned an infinite step so that it comes last.
    """
    if model_name == 'main':
        return float('inf')
    match = re.search(r'\d+', model_name)
    return int(match.group()) if match else float('inf')

def abbreviate_model_name(model_name, index):
    """
    Converts a model name like 'step10000-tokens41B' into a compact label 'S{index}-41B'.
    For example, if index is 1 and model_name is 'step10000-tokens41B', then the label
    will be 'S1-41B'. For 'main', we simply return 'Main'.
    """
    if model_name == 'main':
        return "Main"
    token_match = re.search(r'tokens(\d+B)', model_name)
    token_part = token_match.group(1) if token_match else ""
    return f"S{index}-{token_part}"

def plot_aggregated_proper_heads_iou_agg(iou_results, output_dir):
    """
    Plots aggregated IoU and head count metrics for proper heads across models 
    for a given relation group (e.g. LOC or NAME) in one figure with two subplots.
    Reduces the x ticks (shows every 5th tick) and increases text sizes.
    """
    # Sort model names by training step (with 'main' last)
    model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    
    # Create abbreviated model names for x-axis labels.
    abbreviated_model_names = [abbreviate_model_name(name, idx) 
                               for idx, name in enumerate(model_names, start=1)]
    
    # For the IoU plot, exclude the 'main' model (assumed to be the last element)
    if model_names[-1] == 'main':
        model_names_iou = model_names[:-1]
        abbreviated_model_names_iou = abbreviated_model_names[:-1]
    else:
        model_names_iou = model_names
        abbreviated_model_names_iou = abbreviated_model_names

    # Initialize lists for IoU metrics.
    general_ious = []
    entity_ious = []
    relation_answer_ious = []
    answer_specific_ious = []
    
    # Initialize lists for head counts.
    general_head_count = []
    entity_head_count = []
    relation_answer_head_count = []
    answer_specific_head_count = []
    
    # Populate lists for IoU (skip 'main')
    for model_name in model_names_iou:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        relation_answer_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_specific_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])
    
    # Populate lists for head counts (include all models)
    for model_name in model_names:
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        relation_answer_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_specific_head_count.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])
    
    # Create directories if they do not exist.
    iou_dir = os.path.join(output_dir, "iou")
    count_dir = os.path.join(output_dir, "count")
    os.makedirs(iou_dir, exist_ok=True)
    os.makedirs(count_dir, exist_ok=True)
    
    # Create one figure with 2 subplots (side by side)
    fig, (ax_iou, ax_count) = plt.subplots(nrows=1, ncols=2, figsize=(24, 6))
    
    # ------------------------------
    # Left subplot: Aggregated IoU.
    # ------------------------------
    ax_iou.plot(abbreviated_model_names_iou, general_ious, label="Proper General Heads IoU", marker='o', linestyle='-')
    ax_iou.plot(abbreviated_model_names_iou, entity_ious, label="Proper Entity Heads IoU", marker='s', linestyle='--')
    ax_iou.plot(abbreviated_model_names_iou, relation_answer_ious, label="Proper Relation Answer Heads IoU", marker='x', linestyle='-.')
    ax_iou.plot(abbreviated_model_names_iou, answer_specific_ious, label="Proper Answer Specific Heads IoU", marker='^', linestyle='-.')
    
    # Reduce the number of x ticks: show every 5th tick.
    ticks_iou = range(0, len(abbreviated_model_names_iou), 5)
    labels_iou = [abbreviated_model_names_iou[i] for i in ticks_iou]
    ax_iou.set_xticks(ticks_iou)
    ax_iou.set_xticklabels(labels_iou, rotation=45, ha='right', fontsize=12)
    
    ax_iou.set_xlabel("Model Snapshots", fontsize=14)
    ax_iou.set_ylabel("Aggregated IoU", fontsize=14)
    ax_iou.set_title("Aggregated Proper Heads IoU Across Models", fontsize=16)
    ax_iou.legend(fontsize=12)
    ax_iou.grid(True)
    
    # ------------------------------
    # Right subplot: Aggregated Head Counts.
    # ------------------------------
    ax_count.plot(abbreviated_model_names, general_head_count, label="Proper General Heads Count", marker='o', linestyle='-')
    ax_count.plot(abbreviated_model_names, entity_head_count, label="Proper Entity Heads Count", marker='s', linestyle='--')
    ax_count.plot(abbreviated_model_names, relation_answer_head_count, label="Proper Relation Answer Heads Count", marker='x', linestyle='-.')
    ax_count.plot(abbreviated_model_names, answer_specific_head_count, label="Proper Answer Specific Heads Count", marker='^', linestyle='-.')
    
    # Reduce x ticks: show every 5th tick.
    ticks_count = range(0, len(abbreviated_model_names), 5)
    labels_count = [abbreviated_model_names[i] for i in ticks_count]
    ax_count.set_xticks(ticks_count)
    ax_count.set_xticklabels(labels_count, rotation=45, ha='right', fontsize=12)
    
    ax_count.set_xlabel("Model Snapshots", fontsize=14)
    ax_count.set_ylabel("Aggregated Head Count", fontsize=14)
    ax_count.set_title("Aggregated Proper Heads Count Across Models", fontsize=16)
    ax_count.legend(fontsize=12)
    ax_count.grid(True)
    
    # Adjust spacing between subplots.
    plt.tight_layout()
    
    # Save the combined figure (optional)
    combined_path = os.path.join(output_dir, "combined_aggregated_heads.pdf")
    plt.savefig(combined_path, dpi=500, bbox_inches='tight', format="pdf")
    
    # Display the combined figure.
    plt.show()
    plt.close()

In [ ]:
overlap = calculate_consistency_proper_heads_all_facts_agg(proper_heads_LOC_agg)

# Step 3: Plot aggregated IoU across all facts for the relation
plot_aggregated_proper_heads_iou_agg(overlap, 'final_figures2')
    
    # overlap_NAME = calculate_consistency_proper_heads_all_facts_agg(proper_heads_NAME)

    # # Step 3: Plot aggregated IoU across all facts for the relation
    # plot_aggregated_proper_heads_iou_agg(overlap_NAME, output_dir)

In [ ]:
import numpy as np

def aggregate_answer_specific_heads(data):
    """
    Aggregate proper_answer_specific_heads into a single 32x32 map.
    """
    return np.logical_or.reduce(
        [np.logical_or.reduce(list(fact_maps.values())) for fact_maps in list(data.values())]
    )

def aggregate_relation_answer_heads(data):
    """
    Aggregate proper_relation_answer_heads into a single 32x32 map.
    """
    return np.logical_or.reduce(list(data.values()))

def calculate_switches(proper_heads, revisions, map_types):
    """
    Calculates consistency metrics (e.g. IoU, head counts, role switches) between successive 
    revisions for each map type, using aggregated proper head maps.

    Args:
        proper_heads (dict): Dictionary keyed by revision (e.g., "step5000-20B", "main") where
            each value is a dictionary with keys corresponding to map types:
            'proper_general_heads', 'proper_entity_heads', 
            'proper_relation_answer_heads', 'proper_answer_specific_heads'.
        revisions (list): Sorted list of revision keys.
        map_types (list): List of map type keys.
    
    Returns:
        dict: A dictionary keyed by map type. For each map type, a dictionary is returned where keys are 
              revision pair strings ("rev1 → rev2") and values are dictionaries with metrics:
              - "total_heads_rev1": total head count in rev1 (aggregated)
              - "total_heads_rev2": total head count in rev2
              - "true_to_false": count of heads that transition from True to False
              - "deactivated_count": true-to-false transitions not explained by role switches
              - "new_heads": count of heads that transition from False to True, not stemming from other roles
              - "role_switch_counts": contribution counts from other roles (per map type category)
              - "role_stemming_counts": similar counts for stemming
              - "switched_indices": indices (positions) of heads that switched
              - "stemmed_indices": indices of heads that stemmed from other roles
              - "deactivated_indices": indices of heads that are deactivated
              - "new_heads_indices": indices of new heads.
    """
    results = {map_type: {} for map_type in map_types}
    
    for map_type in map_types:
        for i in range(len(revisions) - 1):
            rev1, rev2 = revisions[i], revisions[i + 1]
            
            # Get data for the current map_type from both revisions.
            data1 = proper_heads[rev1][map_type]
            data2 = proper_heads[rev2][map_type]
            
            # Aggregate maps if necessary.
            # Uncomment and adjust these lines if aggregation is required.
            # if map_type == "proper_answer_specific_heads":
            #     data1_agg = aggregate_answer_specific_heads(data1)
            #     data2_agg = aggregate_answer_specific_heads(data2)
            # elif map_type == "proper_relation_answer_heads":
            #     data1_agg = aggregate_relation_answer_heads(data1)
            #     data2_agg = aggregate_relation_answer_heads(data2)
            # else:
            data1_agg = data1
            data2_agg = data2
            
            # Step 1: Detect transitions.
            true_to_false = np.logical_and(data1_agg, np.logical_not(data2_agg))
            false_to_true = np.logical_and(np.logical_not(data1_agg), data2_agg)
            
            # Step 2: Track role switches.
            switched_into_other_roles = np.zeros_like(true_to_false, dtype=bool)
            stemming_from_other_roles = np.zeros_like(false_to_true, dtype=bool)
            role_switch_counts = {"general": 0, "entity": 0, "relation_answer": 0, "answer_specific": 0}
            role_stemming_counts = {"general": 0, "entity": 0, "relation_answer": 0, "answer_specific": 0}
            switched_indices = []
            stemmed_indices = []
            
            for other_type in map_types:
                if other_type == map_type:
                    continue
                other_data1 = proper_heads[rev1][other_type]
                other_data2 = proper_heads[rev2][other_type]
                
                # Aggregate if needed.
                # if other_type == "proper_answer_specific_heads":
                #     other_data1_agg = aggregate_answer_specific_heads(other_data1)
                #     other_data2_agg = aggregate_answer_specific_heads(other_data2)
                # elif other_type == "proper_relation_answer_heads":
                #     other_data1_agg = aggregate_relation_answer_heads(other_data1)
                #     other_data2_agg = aggregate_relation_answer_heads(other_data2)
                # else:
                other_data1_agg = other_data1
                other_data2_agg = other_data2
                    
                switched_into_other_roles = np.logical_or(
                    switched_into_other_roles,
                    np.logical_and(true_to_false, other_data2_agg)
                )
                stemming_from_other_roles = np.logical_or(
                    stemming_from_other_roles,
                    np.logical_and(false_to_true, other_data1_agg)
                )
                
                # Get indices for switched and stemmed heads.
                switched_mask = np.logical_and(true_to_false, other_data2_agg)
                stemmed_mask = np.logical_and(false_to_true, other_data1_agg)
                switched_indices.extend(list(zip(*np.where(switched_mask))))
                stemmed_indices.extend(list(zip(*np.where(stemmed_mask))))
                
                # Compute contribution counts (using boolean masks).
                role_switch_contribution = np.logical_and(true_to_false, other_data2_agg)
                role_switch_contribution_stemming = np.logical_and(false_to_true, other_data1_agg)
                
                if other_type == "proper_general_heads":
                    role_switch_counts["general"] += np.sum(role_switch_contribution)
                    role_stemming_counts["general"] += np.sum(role_switch_contribution_stemming)
                elif other_type == "proper_entity_heads":
                    role_switch_counts["entity"] += np.sum(role_switch_contribution)
                    role_stemming_counts["entity"] += np.sum(role_switch_contribution_stemming)
                elif other_type == "proper_relation_answer_heads":
                    role_switch_counts["relation_answer"] += np.sum(role_switch_contribution)
                    role_stemming_counts["relation_answer"] += np.sum(role_switch_contribution_stemming)
                elif other_type == "proper_answer_specific_heads":
                    role_switch_counts["answer_specific"] += np.sum(role_switch_contribution)
                    role_stemming_counts["answer_specific"] += np.sum(role_switch_contribution_stemming)
            
            # Step 3: Final deactivated and new heads.
            deactivated = np.logical_and(true_to_false, np.logical_not(switched_into_other_roles))
            new_heads = np.logical_and(false_to_true, np.logical_not(stemming_from_other_roles))
            
            deactivated_count = np.sum(deactivated)
            new_heads_count = np.sum(new_heads)
            
            # Get indices for deactivated and new heads.
            deactivated_indices = list(zip(*np.where(deactivated)))
            new_heads_indices = list(zip(*np.where(new_heads)))
            
            # Store results for this map type and revision pair.
            results[map_type][f"{rev1} → {rev2}"] = {
                "total_heads_rev1": np.sum(data1_agg),
                "total_heads_rev2": np.sum(data2_agg),
                "true_to_false": np.sum(true_to_false),
                "deactivated_count": deactivated_count,
                "new_heads": new_heads_count,
                "role_switch_counts": role_switch_counts,
                "role_stemming_counts": role_stemming_counts,
                "switched_indices": switched_indices,
                "stemmed_indices": stemmed_indices,
                "deactivated_indices": deactivated_indices,
                "new_heads_indices": new_heads_indices,
            }
    
    return results

In [ ]:
revisions = sorted_models
map_types = ["proper_general_heads", "proper_entity_heads", 
             "proper_relation_answer_heads", "proper_answer_specific_heads"]
results_LOC = calculate_switches(proper_heads_LOC_agg, revisions, map_types)

In [ ]:
def plot_switches(results):
    for map_type, map_results in results.items():
        print(f"Results for {map_type}:")
        for change, data in map_results.items():
            print(f"  {change}:")
            print(f"    Total Heads (Rev1): {data['total_heads_rev1']}")
            print(f"    Total Heads (Rev2): {data['total_heads_rev2']}")
            print(f"    True to False: {data['true_to_false']}")
            print(f"    Deactivated Heads: {data['deactivated_count']}")
            print(f"    New Heads: {data['new_heads']}")
            print(f"    Role Switches: {data['role_switch_counts']}")
            print(f"    Role Stemming: {data['role_stemming_counts']}")
            print(f"    Switched heads in {map_type}: {data['switched_indices']}")
            print(f"    Stemmed heads in {map_type}: {data['stemmed_indices']}")
            print(f"    Deactivated heads Indices in: {data['deactivated_indices']}")
            print(f"    New heads Indices in: {data['new_heads_indices']}")
    
plot_switches(results_LOC)

In [ ]:
def compute_head_paths(snapshots):
    """
    Given a dictionary of snapshots, compute for each head (32x32) a list of roles
    over the snapshots.
    
    Parameters:
        snapshots (dict): Keys are snapshot names (e.g., 'step5000-tokens20B' or 'main') and values
                          are dictionaries with keys 'general_heatmap', 'entity_heatmap',
                          'relation_answer_heatmap', 'answer_specific_heatmap' (each a 32x32
                          Boolean numpy array).
    
    Returns:
        paths (dict): A dictionary whose keys are (i, j) tuples (the head positions) and whose
                      values are lists (one per snapshot, in order) giving the role of that head.
                      The role is one of 'general', 'entity', 'relation_answer', 'answer_specific',
                      or 'deactivated'.
    """
    # Separate the 'main' snapshot from the others.
    non_main_keys = [k for k in snapshots if k != "main"]
    # Sort non-main keys based on the numeric value after "step"
    sorted_keys = sorted(non_main_keys, key=lambda k: int(k.split('-')[0].replace('step','')))
    
    # If there is a 'main' snapshot, add it as the last element.
    if "main" in snapshots:
        sorted_keys.append("main")
    
    paths = {}  # to store the path (list of roles) for each head
    # There are 32x32 = 1024 heads.
    for i in range(32):
        for j in range(32):
            head_path = []  # will store the role for this head at each snapshot
            for snap_key in sorted_keys:
                snap = snapshots[snap_key]
                # Determine the role for head (i, j) in this snapshot.
                # Adjust the order if needed. If more than one heatmap is True, this code
                # picks the first one in the order below.
                if snap['proper_general_heads'][i, j]:
                    role = 'general'
                elif snap['proper_entity_heads'][i, j]:
                    role = 'entity'
                elif snap['proper_relation_answer_heads'][i, j]:
                    role = 'relation_answer'
                elif snap['proper_answer_specific_heads'][i, j]:
                    role = 'answer_specific'
                else:
                    role = 'deactivated'
                head_path.append(role)
            paths[(i, j)] = head_path

    return paths

In [ ]:
head_paths = compute_head_paths(proper_heads_LOC_agg)

In [ ]:
import numpy as np
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import pandas as pd
import scipy.cluster.hierarchy as sch

# =============================================================================
# Assume head_paths is already computed.
# For demonstration purposes, we will create a dummy head_paths dictionary.
# In your real analysis, replace this with your actual data.
# =============================================================================

# List of roles
roles = ['general', 'entity', 'relation_answer', 'answer_specific', 'deactivated']

# For demonstration, assume there are 5 revisions (replace with 40 or your actual number).
n_revisions = 40

# =============================================================================
# 1. Markov Chain Modeling
# =============================================================================

def compute_overall_transition_probabilities(head_paths, roles):
    """
    Compute an overall transition probability matrix by summing over all consecutive revision pairs.
    
    Returns:
        P: A numpy array where P[i, j] is the probability of transitioning from role i to role j.
    """
    num_roles = len(roles)
    role_to_idx = {role: i for i, role in enumerate(roles)}
    transition_counts = np.zeros((num_roles, num_roles), dtype=float)
    total_from_counts = np.zeros(num_roles, dtype=int)
    
    for path in head_paths.values():
        for i in range(len(path) - 1):
            src = path[i]
            tgt = path[i + 1]
            s_idx = role_to_idx[src]
            t_idx = role_to_idx[tgt]
            transition_counts[s_idx, t_idx] += 1
            total_from_counts[s_idx] += 1
            
    P = np.zeros_like(transition_counts)
    for i in range(num_roles):
        if total_from_counts[i] > 0:
            P[i, :] = transition_counts[i, :] / total_from_counts[i]
    return P

# Compute the overall transition probability matrix.
P = compute_overall_transition_probabilities(head_paths, roles)

# Print a descriptive table of transitions.
print("Overall Transition Probability Matrix (each cell shows P[From → To]):")
header = "From \\ To\t" + "\t".join(roles)
print(header)
for i, role in enumerate(roles):
    row = role + "\t\t" + "\t".join(f"{P[i, j]:.2f}" for j in range(len(roles)))
    print(row)

plt.figure(figsize=(8, 6))
sns.heatmap(P, annot=True, fmt=".2f", cmap="YlGnBu", xticklabels=roles, yticklabels=roles)
plt.xlabel("Target Role")
plt.ylabel("Source Role")
plt.title("Markov Chain Transition Probability Heatmap")
plt.savefig("final_figures2/head_transition_probability.pdf", dpi=500, bbox_inches='tight', format="pdf")
plt.show()

# -----------------------------------------------------------------------------
# Explanation of the Markov Chain Output:
#
# The printed table shows, for each role (as a source), the probability that a head
# transitions to each target role in the next revision.
#
# For example, if the row for 'general' reads:
# 
#   general       0.50    0.20    0.10    0.15    0.05
#
# It means that, among all heads that are 'general' at some revision, 50% remain 'general'
# in the next revision, 20% become 'entity', 10% become 'relation_answer', 15% become
# 'answer_specific', and 5% become 'deactivated'.
# -----------------------------------------------------------------------------


# -----------------------------------------------------------------------------
# Explanation of the Clustering Output:
#
# The dendrogram is a tree-like diagram that groups head paths based on their similarity.
# - Each leaf (or terminal node) represents an individual head (or a small cluster if truncated).
# - The horizontal axis shows the cluster labels (or clusters when truncated).
# - The vertical axis shows the "linkage distance" (or dissimilarity) at which clusters are merged.
#   Lower distances mean that the merged clusters (or paths) are more similar.
#
# You can use this dendrogram to decide on a cutoff (a distance threshold) to form clusters of similar head paths.
# For example, if you cut the dendrogram at a certain distance, all heads that merge below that distance
# are considered to have similar trajectories.
# -----------------------------------------------------------------------------

# =============================================================================
# 3. Temporal Role Prevalence
# =============================================================================

def compute_role_prevalence(head_paths, roles, n_revisions):
    """
    For each revision, count how many heads are in each role.
    
    Returns:
        A pandas DataFrame with revisions as rows and roles as columns.
    """
    counts = {role: [] for role in roles}
    for rev in range(n_revisions):
        rev_counts = Counter(path[rev] for path in head_paths.values())
        for role in roles:
            counts[role].append(rev_counts.get(role, 0))
    df = pd.DataFrame(counts)
    df.index = [f"Rev {i+1}" for i in range(n_revisions)]
    return df

df_prevalence = compute_role_prevalence(head_paths_LOC, roles, n_revisions)

# Plot a stacked bar chart.
plt.figure(figsize=(10, 6))
df_prevalence.plot(kind="bar", stacked=True)
plt.title("Temporal Role Prevalence Over Revisions")
plt.xlabel("Revision")
plt.ylabel("Number of Heads")
plt.legend(title="Role", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("final_figures2/head_transition_probability.pdf", dpi=500, bbox_inches='tight', format="pdf")
plt.show()



# -----------------------------------------------------------------------------
# Explanation of the Prevalence Plot:
#
# The stacked bar chart shows, for each revision (x-axis), the total number of heads (y-axis)
# broken down by their role.
#
# - Each bar represents one revision (e.g., Rev 1, Rev 2, etc.).
# - Within each bar, each color segment corresponds to one role (e.g., 'general', 'entity', etc.).
# - The height of each segment shows how many heads are in that role at that revision.
#
# This plot lets you see the overall dynamics:
# - Which roles are most prevalent at each revision.
# - Whether there is a shift over time (e.g., a decrease in 'general' heads and an increase in 'entity' heads).
#
# For example, if the 'general' segment of the bar for Rev 1 is very high and then shrinks in later revisions,
# that indicates that many heads start as 'general' but later transition to other roles.
# -----------------------------------------------------------------------------

In [ ]:
def plot_switched_heads_heatmaps_uniform(switched_head_counts_per_map, map_order):
    """
    Creates subplots of 32×32 heatmaps for switched head counts for each map type,
    using a uniform color scale across subplots arranged in a 2x2 grid.
    
    Args:
        switched_head_counts_per_map (dict): A dictionary where each key is a map type (from map_order)
            and its value is a Counter with keys as (layer, head) tuples and values as the number of switches.
        map_order (list): List of map type names in the desired order for subplots.
    """
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    # Convert the counters into a DataFrame.
    plot_data = []
    for map_type in map_order:
        counter = switched_head_counts_per_map[map_type]
        for (layer, head), count in counter.items():
            plot_data.append((map_type, layer, head, count))
    df = pd.DataFrame(plot_data, columns=["Map Type", "Layer", "Head", "Switches"])
    
    # Determine global vmin and vmax for the color scale.
    global_vmin = 0
    global_vmax = df["Switches"].max()
    
    # Create subplots arranged in a 2x2 grid.
    num_maps = len(map_order)
    num_rows, num_cols = 2, 2
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 12), sharex=True, sharey=True)
    axes = axes.flatten()  # Flatten to easily iterate over them
    
    # Define desired font sizes.
    title_fs = 14
    label_fs = 14
    tick_fs  = 12

    for ax, map_type in zip(axes, map_order):
        data = df[df["Map Type"] == map_type]
        if not data.empty:
            # Pivot the DataFrame into a 32x32 matrix (layers x heads).
            heatmap_data = data.pivot(index="Layer", columns="Head", values="Switches").fillna(0)
            # Ensure that the matrix has all 32 layers and 32 heads.
            all_layers = list(range(32))
            all_heads = list(range(32))
            heatmap_data = heatmap_data.reindex(index=all_layers, columns=all_heads, fill_value=0)
            
            # Plot the heatmap with the uniform color scale.
            sns.heatmap(heatmap_data, ax=ax, cmap="Blues", annot=False, cbar=True,
                        vmin=global_vmin, vmax=global_vmax)
            # Invert the y-axis so that the lowest layer number appears at the bottom.
            ax.invert_yaxis()
            ax.set_title(f"Switched Heads - {map_type}", fontsize=title_fs)
            ax.set_xlabel("Head Number", fontsize=label_fs)
            ax.set_ylabel("Layer Number", fontsize=label_fs)
            # Set tick parameters for both axes.
            ax.tick_params(axis='both', labelsize=tick_fs)
            ax.set_xticks(range(0, 32, 2))
            ax.set_yticks(range(0, 32, 2))
        else:
            ax.set_visible(False)
    
    # Hide any unused subplots if the number of maps is less than 4.
    for ax in axes[num_maps:]:
        ax.set_visible(False)
    
    plt.tight_layout()
    #plt.savefig("final_figures2/switches_aggregated.pdf", dpi=500, bbox_inches='tight', format="pdf")
    plt.show()

In [ ]:
from collections import Counter

# Define the correct order of map types for subplots
map_order = ["proper_general_heads", "proper_entity_heads", "proper_relation_answer_heads", "proper_answer_specific_heads"]

# Initialize a dictionary to store counters per map_type
switched_head_counts_per_map_LOC = {map_type: Counter() for map_type in map_order}

# Iterate through results to collect switched head counts per map_type
for map_type in results_LOC:
    if map_type in switched_head_counts_per_map_LOC:  # Ensure we only process expected types
        for transition in results_LOC[map_type]:
            switched_heads = results_LOC[map_type][transition]["switched_indices"]
            for layer, head in switched_heads:
                switched_head_counts_per_map_LOC[map_type][(layer, head)] += 1

plot_switched_heads_heatmaps_uniform(switched_head_counts_per_map_LOC, map_order)



# # Initialize a dictionary to store counters per map_type
# switched_head_counts_per_map_NAME = {map_type: Counter() for map_type in map_order}

# # Iterate through results to collect switched head counts per map_type
# for map_type in results_NAME:
#     if map_type in switched_head_counts_per_map_NAME:  # Ensure we only process expected types
#         for transition in results_NAME[map_type]:
#             switched_heads = results_NAME[map_type][transition]["switched_indices"]
#             for layer, head in switched_heads:
#                 switched_head_counts_per_map_NAME[map_type][(layer, head)] += 1

# plot_switched_heads_heatmaps_uniform(switched_head_counts_per_map_NAME, map_order)

In [ ]:
import numpy as np
import pandas as pd

def extract_transitions_by_steps(results, revisions):
    """
    Aggregates the metrics from all transitions that occur between each adjacent
    pair of revisions in the provided list. The metrics for each transition are
    summed (not averaged), and the function returns a dictionary mapping each
    revision pair (as "rev1 → rev2") to a DataFrame containing the aggregated counts.
    
    The output DataFrame has 5 rows (corresponding to:
      - "General Heads"
      - "Entity Heads"
      - "Relation Answer Heads"
      - "Answer Specific Heads"
      - "Deactivated Count" (aggregated from each category's new_heads))
    and 5 columns:
      ["general", "entity", "relation_answer", "answer_specific", "deactivated_count"]
    
    Args:
        results (dict): A dictionary with keys for each proper head category:
                        "proper_general_heads", "proper_entity_heads",
                        "proper_relation_answer_heads", "proper_answer_specific_heads".
                        Each value is itself a dictionary keyed by transition strings (e.g. "step5000-tokens20B → step50000-tokens209B")
                        containing metrics (including "role_switch_counts", "deactivated_count", and "new_heads").
        revisions (list): A list of revision strings (in order), e.g.:
                          ['step5000-tokens20B', 'step50000-tokens209B', 'step100000-tokens419B', 'step200000-tokens838B', 'main'].
                          
    Returns:
        dict: A dictionary mapping each adjacent revision pair (as "rev1 → rev2") to a DataFrame with the aggregated metrics.
    """
    # Helper: extract numeric step from a revision string (treat "main" as infinity)
    def get_step(rev):
        if rev == "main":
            return float("inf")
        try:
            return int(rev.split("step")[1].split("-")[0])
        except Exception as e:
            raise ValueError(f"Cannot parse revision '{rev}': {e}")
    
    # Define the proper head category keys and the metric column names.
    category_keys = ["proper_general_heads", "proper_entity_heads", 
                     "proper_relation_answer_heads", "proper_answer_specific_heads"]
    value_keys = ["general", "entity", "relation_answer", "answer_specific", "deactivated_count"]
    
    # Build the list of transitions for each adjacent pair of revisions.
    transitions_by_pair = {}
    for i in range(len(revisions) - 1):
        lower, upper = revisions[i], revisions[i+1]
        lower_step = get_step(lower)
        upper_step = get_step(upper)
        pair_key = f"{lower} → {upper}"
        transitions = []
        # We'll look at transitions from the first category (assuming all categories share the same transition keys)
        # (If not, you might want to combine keys from all categories.)
        sample_cat = category_keys[0]
        if sample_cat in results:
            for trans in results[sample_cat].keys():
                try:
                    rev1, rev2 = trans.split(" → ")
                    step1 = get_step(rev1)
                    step2 = get_step(rev2)
                    if step1 >= lower_step and step2 <= upper_step:
                        transitions.append(trans)
                except Exception:
                    pass
        transitions_by_pair[pair_key] = transitions
    
    # Now, for each adjacent revision pair, aggregate the metrics from all transitions that fall in that interval.
    aggregated_results = {}
    for pair_key, trans_list in transitions_by_pair.items():
        # Initialize an aggregated matrix with shape (number of categories + 1, number of value keys).
        # Rows 0-3: the four proper head categories.
        # Row 4: will be used to aggregate "new_heads" counts.
        aggregated_matrix = np.zeros((len(category_keys) + 1, len(value_keys)))
        for trans in trans_list:
            # For each transition, build a matrix with the same shape.
            matrix = np.zeros((len(category_keys) + 1, len(value_keys)))
            for i, cat in enumerate(category_keys):
                total_counts = {key: 0 for key in value_keys}
                if trans in results.get(cat, {}):
                    # Update with the role switch counts.
                    total_counts.update(results[cat][trans].get("role_switch_counts", {}))
                    # Add the deactivated count.
                    total_counts["deactivated_count"] += results[cat][trans].get("deactivated_count", 0)
                matrix[i] = [total_counts[v] for v in value_keys]
            # For the last row, aggregate "new_heads" counts from each category (except you can exclude one if desired)
            new_heads_counts = {key: 0 for key in value_keys}
            for i, cat in enumerate(category_keys):  # Here we consider the first three categories, for example.
                if trans in results.get(cat, {}):
                    new_heads_counts[value_keys[i]] += results[cat][trans].get("new_heads", 0)
            matrix[-1] = [new_heads_counts[v] for v in value_keys]
            aggregated_matrix += matrix
        # Do not average; we want the total.
        df_extracted = pd.DataFrame(
            aggregated_matrix,
            index=["General Heads", "Entity Heads", "Relation Answer Heads", "Answer Specific Heads", "Deactivated Count"],
            columns=value_keys
        )
        aggregated_results[pair_key] = df_extracted
    return aggregated_results

# Example usage:
revisions = ['step5000-tokens20B', 'step50000-tokens209B', 'step100000-tokens419B',
             'step200000-tokens838B', 'main']
# Suppose results_LOC is your results dictionary for LOC.
aggregated_dfs_LOC = extract_transitions_by_steps(results_LOC, revisions)
#aggregated_dfs_NAME = extract_transitions_by_steps(results_NAME, revisions)
# Now, aggregated_dfs_LOC is a dictionary where keys are transitions, e.g.:
# "step5000-tokens20B → step50000-tokens209B", "step50000-tokens209B → step100000-tokens419B", etc.
# Each value is a DataFrame with the aggregated counts.
for key, df in aggregated_dfs_LOC.items():
    print(f"Transition: {key}")
    print(df)


In [ ]:
import math
import re
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Updated helper functions.
def extract_step_number(model_name):
    """
    Extracts the step number from the model name using regex.
    'main' is assigned an infinite step so that it comes last.
    """
    if model_name == 'main':
        return float('inf')
    match = re.search(r'\d+', model_name)
    return int(match.group()) if match else float('inf')

def abbreviate_model_name(model_name, index):
    """
    Converts a model name like 'step10000-tokens41B' into a compact label 'SX-YB'
    where X is the step number divided by 5000.
    For example, 'step10000-tokens41B' becomes 'S2-41B'.
    For 'main', returns 'Main'.
    """
    if model_name == 'main':
        return "Main"
    step_match = re.search(r'step(\d+)', model_name)
    if step_match:
        step_num = int(step_match.group(1))
        s_num = step_num // 5000
    else:
        s_num = index
    token_match = re.search(r'tokens(\d+B)', model_name)
    token_part = token_match.group(1) if token_match else ""
    return f"S{s_num}-{token_part}"

def plot_all_heatmaps(extracted_matrices):
    """
    Creates subplots of 32×32 heatmaps for switched head counts for each map type,
    using a uniform color scale across subplots arranged in a 2x2 grid.
    
    The models are first sorted and only the first five are selected.
    Each subplot's title shows an abbreviated range (e.g. "S1-20B → S2-41B")
    if a next model exists. Only the outer subplots display tick labels, and a common
    x-axis label ("Categories") is added.
    
    The y-axis label ("Head Types") is set only on the leftmost subplots.
    
    Font sizes have been increased for better readability.
    
    Args:
        extracted_matrices (dict): A dictionary where each key is a model name (or transition)
            and its value is a DataFrame (or matrix) representing the heatmap.
    """
    # Sort model names using extract_step_number.
    sorted_model_names = sorted(extracted_matrices.keys(), key=lambda x: extract_step_number(x))
    # Select only the first five models.
    selected_model_names = sorted_model_names[:5]
    num_matrices = len(selected_model_names)
    cols = 2  # Number of heatmaps per row
    rows = math.ceil(num_matrices / cols)

    # Define your manual titles.
    manual_titles = [
        "S1-20B -> S10-209B",
        "S10-209B -> S20-419B",
        "S20-419B -> S40-838B",
        "S40-838B -> Main"
    ]

    # Removed sharey so that each left subplot can have its own y-axis label.
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 8, rows * 8), sharex=True)
    axes = axes.flatten()  # Flatten in case we don't get a perfect grid

    # Increase font sizes.
    title_fs = 14
    label_fs = 18
    tick_fs  = 14

    for i, model_name in enumerate(selected_model_names):
        df_matrix = extracted_matrices[model_name]
        
        # Use manual title if available; otherwise, generate one automatically.
        if i < len(manual_titles):
            short_title = manual_titles[i]
        else:
            # Generate title: if there's a next model, display "current → next"; otherwise, just current.
            if i < len(selected_model_names) - 1:
                next_model = selected_model_names[i+1]
                short_title = f"{abbreviate_model_name(model_name, i+1)} → {abbreviate_model_name(next_model, i+2)}"
            else:
                short_title = abbreviate_model_name(model_name, i+1)
        
        # Plot the heatmap.
        sns.heatmap(df_matrix.astype(int), annot=True, fmt="d", cmap="coolwarm", linewidths=0.5, ax=axes[i])
        axes[i].set_title(short_title, fontsize=title_fs)
        
        # Remove individual x axis label.
        axes[i].set_xlabel("")
        # Remove individual y axis label (we'll add it for left subplots).
        axes[i].set_ylabel("")
        
        # Determine grid position.
        row_pos = i // cols
        col_pos = i % cols
        
        # Only show x tick labels on bottom row.
        if row_pos == rows - 1:
            axes[i].tick_params(axis='x', labelsize=tick_fs)
        else:
            axes[i].set_xticklabels([])
        
        # Only show y tick labels on left column.
        if col_pos == 0:
            axes[i].tick_params(axis='y', labelsize=tick_fs)
            # Set the left y-axis label.
            axes[i].set_ylabel("Head Types", fontsize=label_fs)
        else:
            axes[i].set_yticklabels([])

    # Hide any unused subplots.
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Add a common x-axis label.
    fig.text(0.5, 0.04, "Categories", ha="center", fontsize=label_fs)
    
    plt.tight_layout(rect=[0.08, 0.08, 1, 1])
    plt.savefig("final_figures2/switch_count_aggregated_2.pdf", bbox_inches="tight", dpi=500, format="pdf")
    plt.show()
    
plot_all_heatmaps(aggregated_dfs_LOC)